# Fase 6 — Ambiente Gymnasium

Implementa l'ambiente RL M15 con API Gymnasium. L'osservazione contiene le 13 feature di mercato della Fase 3 più current_position e unrealized_return. L'azione viene eseguita sulla barra successiva rispetto a quella osservata, così il Close usato per costruire l'osservazione non viene riutilizzato per l'esecuzione.

Checklist: Discrete(3), reset, step, protezione dagli accessi oltre indice, episodi random solo in training, segmenti cronologici in validation/test, check_env, random agent end-to-end.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import yaml
from gymnasium.utils.env_checker import check_env

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from gold_rl.data.features import MARKET_FEATURE_COLUMNS
from gold_rl.execution import ExecutionCostsConfig
from gold_rl.trading_env import TradingEnv

config = yaml.safe_load((PROJECT_ROOT / 'config.yaml').read_text())
environment = config['environment']
print('Feature attese:', MARKET_FEATURE_COLUMNS)
print('Initial balance:', environment['initial_balance'])

## 1. Input M15

Il notebook non ricostruisce feature da M1: l'ambiente riceve direttamente un DataFrame M15 già costruito dalla Fase 3. Vengono provati alcuni nomi convenzionali di parquet.

In [ ]:
candidates = [
    PROJECT_ROOT / 'data/processed/xauusd_m15_features.parquet',
    PROJECT_ROOT / 'data/processed/xauusd_m15.parquet',
    PROJECT_ROOT / 'data/processed/m15_features.parquet',
]
path = next((p for p in candidates if p.exists()), None)
if path is None:
    raise FileNotFoundError('Nessun parquet M15 trovato in data/processed. Usa il prodotto della Fase 3.')

data = pd.read_parquet(path)
if not isinstance(data.index, pd.DatetimeIndex):
    raise TypeError('Il parquet M15 deve avere un DatetimeIndex.')

missing = [c for c in MARKET_FEATURE_COLUMNS if c not in data.columns]
if missing:
    raise ValueError(f'Feature M15 mancanti: {missing}')
if 'Close' not in data.columns:
    raise ValueError('Il dataset passato all ambiente deve contenere anche Close.')

features = data.loc[:, MARKET_FEATURE_COLUMNS].copy()
prices = data['Close'].copy()
features = features.replace([np.inf, -np.inf], np.nan).dropna()
prices = prices.reindex(features.index)
assert len(features) >= 2
print(path, len(features), features.index.min(), features.index.max())

## 2. Costi ed environment

L'ambiente riusa execute_order e update_portfolio della Fase 4: non viene creato un secondo execution engine. Il reward è il rendimento netto dell'equity; turnover_reward_weight consente un'ulteriore penalizzazione del turnover come reward shaping.

In [ ]:
costs = ExecutionCostsConfig(
    spread_rate=float(environment['spread_rate']),
    commission_rate=float(environment['commission_rate']),
    slippage_rate=float(environment['slippage_rate']),
    turnover_penalty=float(environment['turnover_penalty']),
)

train_env = TradingEnv(features=features, mid_prices=prices, execution_config=costs, initial_balance=float(environment['initial_balance']), mode='train', max_episode_steps=int(environment['max_episode_steps']), turnover_reward_weight=0.0)
validation_env = TradingEnv(features=features, mid_prices=prices, execution_config=costs, initial_balance=float(environment['initial_balance']), mode='validation', max_episode_steps=None, random_start=False)
test_env = TradingEnv(features=features, mid_prices=prices, execution_config=costs, initial_balance=float(environment['initial_balance']), mode='test', max_episode_steps=None, random_start=False)

print(train_env.action_space)
print(train_env.observation_space)

## 3. Gymnasium check_env

La verifica controlla API, spazi, reset, seed e step. È importante eseguirla sull'environment raw.

In [ ]:
check_env(train_env, skip_render_check=True)
print('check_env: OK')

## 4. Random agent — episodio completo

Il test percorre l'intero episodio con action_space.sample() e verifica che osservazioni, reward ed equity rimangano finite.

In [ ]:
obs, info = train_env.reset(seed=int(config['project']['seed']))
steps = 0
while True:
    assert np.isfinite(obs).all()
    action = train_env.action_space.sample()
    obs, reward, terminated, truncated, info = train_env.step(action)
    assert np.isfinite(obs).all()
    assert np.isfinite(reward)
    assert np.isfinite(info['equity'])
    steps += 1
    if terminated or truncated:
        break
print('Random episode steps:', steps)
print('Final equity:', info['equity'])

## 5. Segmenti validation/test

Validation e test non usano random start: reset parte dall'inizio del segmento e step avanza esclusivamente in ordine cronologico. Per evitare look-ahead, l'azione osservata su t viene eseguita a t+1.

In [ ]:
for name, env in [('validation', validation_env), ('test', test_env)]:
    _, info = env.reset(seed=123)
    first = info['timestamp']
    _, _, terminated, truncated, info = env.step(1)
    second = info['timestamp']
    assert second > first
    assert not truncated
    print(name, first, '->', second)

print('Fase 6 completata.')